# TL;DR

The completed gate-branch review does **not** establish that 6 mm is worse
than 8 mm. Human female identity and automatic SLEAP/geometry mapping quality
must be scored separately. Until the new structured evidence-quality field is
reviewed, the automatic threshold comparison remains inconclusive.


In [1]:
from pathlib import Path
import importlib.util
import json
import pandas as pd

audit_path = Path('report/20260718/k12_jump6_strict_mouth/gate_branch_diagnostic/audit_review_and_motion.py').resolve()
spec = importlib.util.spec_from_file_location('gate_motion_audit', audit_path)
audit = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(audit)
result = audit.run()
result['summary']

{'schema_version': 1,
 'decision': '6_vs_8_inconclusive_pending_quality_review',
 'review': {'branches': 26,
  'annotated_branches': 25,
  'two_view_consistent_candidates': 14,
  'explicitly_quality_reviewed': 0,
  'tracker_scoring_eligible': 0,
  'policy': 'Only evidence_quality=usable may score an automatic geometry mapping; human_only remains valid human identity evidence.'},
 'motion': {'changed_track_boundaries': 53,
  'history_policy': '15 direct observations on each side within 30 frames; Theil-Sen-style median of all pairwise 3D slopes',
  'jump6_tangent_angle_median_deg': 35.65152875493839,
  'jump8_tangent_angle_median_deg': 39.65580410776083,
  'jump6_boundaries_over_90_deg': 5,
  'jump8_boundaries_over_90_deg': 6,
  'warning': 'Motion continuity is a diagnostic, not identity truth, when SLEAP observations are false-negative, shared, or merged.'}}

# Context & Methods

The 6 mm and controlled 8 mm reconstructions use the same per-frame 3D point
sets; their difference is temporal linking. The original linker uses only the
last accepted position because the optional velocity/tangent/acceleration
weights were zero.

This audit uses two independent checks:

1. Human evidence is eligible for tracker scoring only when the female is
   confirmed, both side views link to SLEAP, both views agree on geometry, and
   `evidence_quality=usable`. A `human_only` scene remains valuable human
   identity evidence but its SLEAP/geometry mapping is excluded.
2. Motion continuity is measured bidirectionally using up to 15 direct
   observations in the 30 frames before and after each branch. Velocity is the
   median of all pairwise 3D slopes, so the result uses more than two frames
   and is resistant to one tangent spike.

Motion is diagnostic evidence only. It cannot turn a false negative, shared
tail/node, or merged SLEAP instance into a valid observation.


In [2]:
pd.DataFrame({
    'input': ['human ledger', '6 mm track', '8 mm controlled track', 'branch transitions'],
    'path': [str(audit.LEDGER), str(audit.JUMP6), str(audit.JUMP8), str(audit.TRANSITIONS)],
})

,input,path
0,human ledger,C:\Users\machi\Documents\GitHub\medaka-mating-...
1,6 mm track,C:\Users\machi\Documents\GitHub\medaka-mating-...
2,8 mm controlled track,C:\Users\machi\Documents\GitHub\medaka-mating-...
3,branch transitions,C:\Users\machi\Documents\GitHub\medaka-mating-...


# Data

In [3]:
quality = result['quality']
quality_counts = pd.DataFrame({
    'measure': [
        'branches', 'annotated branches', 'two-view-consistent candidates',
        'explicit quality reviews', 'tracker-scoring eligible'
    ],
    'count': [
        len(quality), quality.annotation_frame.notna().sum(),
        quality.automatic_mapping_candidate.sum(),
        quality.evidence_quality.ne('unreviewed').sum(),
        quality.tracker_scoring_eligible.sum(),
    ],
})
quality_counts

,measure,count
0,branches,26
1,annotated branches,25
2,two-view-consistent candidates,14
3,explicit quality reviews,0
4,tracker-scoring eligible,0


In [4]:
quality[[
    'branch_id', 'trigger_frame', 'annotation_frame', 'gt_status',
    'side1_selection_status', 'side2_selection_status',
    'geometry_id_consistency', 'evidence_quality',
    'observation_issue_flags', 'automatic_mapping_candidate',
    'tracker_scoring_eligible', 'notes'
]]

,branch_id,trigger_frame,annotation_frame,gt_status,side1_selection_status,side2_selection_status,geometry_id_consistency,evidence_quality,observation_issue_flags,automatic_mapping_candidate,tracker_scoring_eligible,notes
0,B001,18324,18324,confirmed,sleap_linked,sleap_linked,agree,unreviewed,,True,False,
1,B002,18366,18366,confirmed,sleap_linked,sleap_linked,agree,unreviewed,,True,False,
2,B003,18847,18847,confirmed,sleap_linked,sleap_linked,disagree,unreviewed,,False,False,
3,B004,19537,19537,confirmed,sleap_linked,not_selected,one_view,unreviewed,,False,False,
4,B005,27490,27490,confirmed,sleap_linked,sleap_linked,agree,unreviewed,,True,False,
5,B006,30812,30812,confirmed,sleap_linked,sleap_linked,agree,unreviewed,,True,False,
6,B007,36839,36839,confirmed,sleap_linked,sleap_linked,disagree,unreviewed,,False,False,
7,B008,74757,74757,confirmed,sleap_linked,sleap_linked,agree,unreviewed,,True,False,
8,B009,89328,89328,confirmed,sleap_linked,sleap_linked,agree,unreviewed,,True,False,
9,B010,89391,89391,confirmed,sleap_linked,sleap_linked,agree,unreviewed,,True,False,


# Results

In [5]:
motion = result['motion']
paired = motion.dropna(subset=[
    'jump6_robust_tangent_angle_deg', 'jump8_robust_tangent_angle_deg'
]).copy()
pd.DataFrame({
    'measure': ['paired changed-track boundaries', '6 mm lower tangent', '8 mm lower tangent',
                '6 mm tangent >90°', '8 mm tangent >90°'],
    'count': [len(paired),
              (paired.jump6_robust_tangent_angle_deg < paired.jump8_robust_tangent_angle_deg).sum(),
              (paired.jump8_robust_tangent_angle_deg < paired.jump6_robust_tangent_angle_deg).sum(),
              (paired.jump6_robust_tangent_angle_deg > 90).sum(),
              (paired.jump8_robust_tangent_angle_deg > 90).sum()],
})

,measure,count
0,paired changed-track boundaries,53
1,6 mm lower tangent,32
2,8 mm lower tangent,21
3,6 mm tangent >90°,5
4,8 mm tangent >90°,6


In [6]:
paired.assign(
    max_tangent_deg=paired[[
        'jump6_robust_tangent_angle_deg', 'jump8_robust_tangent_angle_deg'
    ]].max(axis=1)
).sort_values('max_tangent_deg', ascending=False)[[
    'branch_id', 'trigger_frame', 'six_id', 'old_eight_id', 'new_eight_id',
    'jump6_robust_tangent_angle_deg', 'jump8_robust_tangent_angle_deg',
    'jump6_forward_bridge_prediction_error_mm',
    'jump8_forward_bridge_prediction_error_mm'
]].head(20)

,branch_id,trigger_frame,six_id,old_eight_id,new_eight_id,jump6_robust_tangent_angle_deg,jump8_robust_tangent_angle_deg,jump6_forward_bridge_prediction_error_mm,jump8_forward_bridge_prediction_error_mm
47,B024,213207,ID_1,ID_2,ID_3,60.259709,151.576619,0.377187,7.196150
48,B024,213207,ID_2,ID_3,ID_2,95.112975,139.433871,10.837979,6.532098
11,B006,30812,ID_3,ID_2,ID_1,134.324192,42.646826,32.671233,6.824906
36,B018,133887,ID_3,ID_1,ID_3,134.245763,72.179535,0.574875,3.996922
24,B013,95368,ID_2,ID_3,ID_1,41.901696,124.310985,11.043813,7.374205
44,B022,134488,ID_3,ID_3,ID_1,113.258483,76.436658,2.676734,7.503644
10,B006,30812,ID_1,ID_1,ID_2,46.372304,108.592698,0.285753,0.285753
28,B015,126465,ID_1,ID_3,ID_2,108.424020,99.873245,11.069409,7.478629
32,B017,133848,ID_1,ID_1,ID_2,29.903146,94.937795,11.407557,6.250510
35,B018,133887,ID_2,ID_3,ID_1,48.391646,89.083203,8.558905,7.013763


# Takeaways

- The previous recommendation to prefer 8 mm is withdrawn: the new review
  demonstrates an observation-quality confound, and none of the legacy rows
  has yet been explicitly marked usable for tracker scoring.
- Robust multi-frame tangent evidence is mixed rather than uniformly favoring
  one gate. Large tangent discontinuities identify useful human-review scenes,
  but they are not ground truth when SLEAP itself is wrong.
- Use 8 mm (with the existing 80 mm hard cap) only as an outer candidate gate
  while testing motion-aware association. The identity choice should use
  robust history and eventually short lookahead; 6-versus-8 should be retested
  only on explicitly `usable` human evidence.
